# LangChain Mini-Projects Workshop

A hands-on notebook built around the current LangChain Python API and the idea:

> **Agent = Model + Harness**

In other words: an "agent" is not magic. It's a language model wrapped in ordinary
software (the "harness") that gives it tools to call, a way to remember
conversation state, a schema to shape its output, and rules that run before/after
it thinks. This notebook builds that harness up piece by piece so each concept is
isolated and easy to reason about.

The notebook uses `create_agent` and gradually adds tools, structured output, short-term memory, and middleware.

## Projects

1. **Bookstore Concierge** — your first tool-using agent.
   Teaches the most fundamental agent pattern: the model cannot access real data
   on its own, so it must call a Python function ("tool") to look up book
   inventory instead of guessing or hallucinating titles/prices.

2. **Math Detective** — multiple deterministic tools and tool selection.
   Gives the agent several overlapping math tools (calculator, primality test,
   factorization, gcd) so you can observe how the model decides *which* tool
   fits a given question, and how it chains multiple tool calls together to
   answer a multi-part question.

3. **Bug Triage Desk** — structured output with Pydantic.
   Instead of free-form prose, the agent is forced to return data that matches
   a strict schema (severity, component, reproduction steps, etc.), which is
   what real backends need to route work automatically.

4. **Tiny Escape Room** — an agent acting on a small simulated world.
   The model can only change the "world" (a dictionary of room state) by
   calling tools that check preconditions in plain Python. This shows how to
   prevent an agent from ever being able to fake success.

5. **Study Coach with Memory** — multi-turn state with a checkpointer.
   Demonstrates that, by default, every agent call is stateless. Adding a
   checkpointer keyed by `thread_id` lets the agent remember earlier turns in
   the same conversation, without a database.

6. **Guarded Support Agent** — middleware around the agent loop.
   Shows how to intercept and modify what the model sees *before* it runs,
   using a `before_model` hook that trims conversation history — a simple
   example of the more general idea that policy (safety, cost control,
   logging) belongs in code around the model, not just in prompt wording.

Each project is intentionally small enough to understand in one sitting.

Official docs used as the basis:
- https://docs.langchain.com/oss/python/langchain/overview
- https://docs.langchain.com/oss/python/langchain/agents
- https://docs.langchain.com/oss/python/langchain/tools
- https://docs.langchain.com/oss/python/langchain/structured-output
- https://docs.langchain.com/oss/python/langchain/short-term-memory
- https://docs.langchain.com/oss/python/langchain/middleware

## 0. Setup

This notebook defaults to OpenAI, but the rest of the code is written so that you can replace `MODEL` with another LangChain-supported provider model.

Set your API key in the environment before running the agent cells.

For OpenAI, for example:

```text
OPENAI_API_KEY=...
```

LangSmith tracing is optional.

**Note:** the setup cell below actually configures **Groq** (via a `.env` file
and `GROQ_API_KEY`), not OpenAI — LangChain's `MODEL` string just needs a
`provider:model-name` format, so switching providers is a one-line change.

In [1]:
%pip install -qU langchain "langchain[openai]" langgraph pydantic

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
investiny 0.7.2 requires httpx<0.24.0,>=0.23.0, but you have httpx 0.28.1 which is incompatible.
investiny 0.7.2 requires pydantic<2.0.0,>=1.10.2, but you have pydantic 2.13.5 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import random
import json
from datetime import date, timedelta
from typing import Literal

# python-dotenv lets us keep secrets (API keys) out of the notebook itself.
# It reads a local ".env" file and loads its key=value pairs into os.environ.
from dotenv import load_dotenv
load_dotenv()  # reads GROQ_API_KEY from .env

# Fail fast with a clear error instead of letting the agent calls fail later
# with a confusing authentication exception.
if not os.environ.get("GROQ_API_KEY"):
    raise RuntimeError("Set GROQ_API_KEY in your .env file")

# LangChain model strings use the format "provider:model-name".
# Here we use Groq's hosted, fast inference of an open-weight GPT-OSS model.
# Swap this string to use a different provider/model without touching any
# other code below (e.g. "openai:gpt-4o-mini" or "anthropic:claude-sonnet-5").
MODEL = "groq:openai/gpt-oss-20b"  # swap for a cheaper/bigger Groq model if you like

# create_agent: the main LangChain factory that wires a model + tools +
#   prompt + (optionally) memory/middleware into a runnable agent.
# AgentState: the TypedDict describing what data flows through the agent
#   graph (mainly the running list of messages).
# tool: decorator that turns a plain Python function into something the
#   model can discover and call, using its type hints/docstring as the
#   function's name/parameter schema/description.
from langchain.agents import create_agent, AgentState
from langchain.tools import tool
# Pydantic BaseModel/Field are used later to define structured-output schemas.
from pydantic import BaseModel, Field

print("Ready. Model:", MODEL)

Ready. Model: groq:openai/gpt-oss-20b


In [3]:
# (Re-imported here for clarity/redundancy in case this cell is run standalone.)
import os
from pprint import pprint  # pretty-prints nested dicts/lists (used for ROOM state later)

from langchain.agents import create_agent
from langchain.tools import tool


In [4]:
def ask(agent, message, config=None):
    """Small helper for notebook demos.

    Wraps the slightly verbose LangChain invocation call so every project
    below can just call `ask(agent, "some question")`.

    - `agent.invoke(...)` runs the full agent loop synchronously: the model
      may call tools zero or more times, and the loop keeps going until the
      model produces a final answer (or, for structured-output agents, a
      validated object).
    - The input must be a dict with a "messages" list, matching the chat
      message format LangChain/most LLM APIs expect.
    - `config` is how you pass a `thread_id` for conversation memory
      (see Project 5); without it, each call is independent/stateless.
    """
    payload = {"messages": [{"role": "user", "content": message}]}
    if config is None:
        result = agent.invoke(payload)
    else:
        result = agent.invoke(payload, config=config)

    # The agent's full message history is returned; the last message is
    # the model's final natural-language reply (tool-call messages and their
    # results are earlier entries in the list).
    final_message = result["messages"][-1]
    print(final_message.content)
    return result

# 1. Bookstore Concierge

### What you learn
- `create_agent`
- Python functions as tools
- Tool descriptions as part of the agent interface
- The difference between **LLM reasoning** and **deterministic application logic**

The model does **not** know our shop inventory. It must call tools to inspect it.

### How it works, step by step

1. We define a small, hard-coded `BOOKS` list — this stands in for a real
   database or inventory API.
2. We wrap two plain Python functions (`search_books`, `book_details`) with
   the `@tool` decorator. This is what actually exposes them to the model:
   LangChain reads each function's **name**, **docstring**, and **type
   hints** and turns them into a schema the model can read (like a tiny API
   spec) and a JSON-callable interface at runtime.
3. `create_agent(model=..., tools=[...], system_prompt=...)` builds the
   agent. The `system_prompt` is instructions that persist across the whole
   conversation — here it explicitly tells the model *not* to invent
   inventory, forcing it to rely on the tools instead of its own guesses.
4. When you call `ask(book_agent, "...")`, the model reads the user's
   question, decides a tool call is needed, LangChain executes the matching
   Python function with the arguments the model chose, feeds the tool's
   return value back to the model as a new message, and the model then
   composes the final natural-language answer.

This "look before you answer" pattern is the foundation every other project
in this notebook builds on.

In [5]:
# Hard-coded "database" of book inventory. In a real app this would be a
# call to a DB or search index; the agent must never see this data directly,
# only through the tool functions below.
BOOKS = [
    {"title": "The Left Hand of Darkness", "author": "Ursula K. Le Guin", "genre": "science fiction", "price": 14.0, "rating": 4.6},
    {"title": "The Dispossessed", "author": "Ursula K. Le Guin", "genre": "science fiction", "price": 12.0, "rating": 4.5},
    {"title": "Piranesi", "author": "Susanna Clarke", "genre": "fantasy", "price": 11.0, "rating": 4.4},
    {"title": "The City & the City", "author": "China Miéville", "genre": "mystery", "price": 13.5, "rating": 4.2},
    {"title": "Gödel, Escher, Bach", "author": "Douglas Hofstadter", "genre": "science", "price": 22.0, "rating": 4.3},
    {"title": "Chaos", "author": "James Gleick", "genre": "science", "price": 10.5, "rating": 4.1},
]

@tool
def search_books(genre: str = "", max_price: float = 10_000) -> str:
    """Search the local bookstore catalog by genre and maximum price."""
    # NOTE: the text above (the docstring) is not just documentation for
    # humans — it's sent to the model as the tool's description, so the
    # model knows *when* and *how* to call this function.
    matches = [
        b for b in BOOKS
        # empty genre string means "no filter"; substring match is
        # case-insensitive so "sci" or "Science Fiction" both work
        if (not genre or genre.lower() in b["genre"].lower())
        and b["price"] <= max_price
    ]
    if not matches:
        return "No matching books."
    # Tools must return something the model can read — plain text or JSON,
    # not Python objects — so we format each match as a readable line.
    return "\n".join(
        f'{b["title"]} — {b["author"]} | {b["genre"]} | ${b["price"]:.2f} | rating {b["rating"]}'
        for b in matches
    )

@tool
def book_details(title: str) -> str:
    """Get exact information about one book by title."""
    for b in BOOKS:
        if b["title"].lower() == title.lower():
            return str(b)
    return "Book not found."

# Assemble the agent: a model, the list of tools it's allowed to call, and
# a system prompt steering its behavior. This single call is what turns a
# plain chat model into a tool-using agent.
book_agent = create_agent(
    model=MODEL,
    tools=[search_books, book_details],
    system_prompt=(
        "You are a bookstore concierge. Never invent inventory. "
        "Use the catalog tools before recommending a book."
    ),
)

In [7]:
ask(
    book_agent,
    "I want a science-fiction book under $13. Recommend one and explain why it fits."
)

**Recommended book:** *The Dispossessed* by Ursula K. Le Guin  
**Price:** $12.00  
**Genre:** Science Fiction  
**Rating:** 4.5 ★  

**Why it fits your criteria**

- **Budget‑friendly:** The book is priced at $12, comfortably below your $13 limit.  
- **Science‑fiction core:** It’s a classic SF novel that blends speculative world‑building with deep philosophical questions.  
- **Engaging narrative:** Le Guin’s storytelling balances vivid, imaginative settings with an insightful exploration of anarchism, freedom, and human connection—perfect for readers who enjoy thoughtful, thought‑provoking sci‑fi.  

Let me know if you’d like to place an order or need more details!


{'messages': [HumanMessage(content='I want a science-fiction book under $13. Recommend one and explain why it fits.', additional_kwargs={}, response_metadata={}, id='9fff8568-8cf4-41cd-a035-6fc5a4b06f95'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We must use the catalog tools. First search books with genre "Science Fiction" and max_price 13. Then choose one. Probably we need to pick a title that exists. We need to call search_books.', 'tool_calls': [{'id': 'fc_ad66d3cc-c31f-4985-b172-16f7b8f65c72', 'function': {'arguments': '{"genre":"Science Fiction","max_price":13}', 'name': 'search_books'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 204, 'total_tokens': 277, 'completion_time': 0.084486935, 'completion_tokens_details': {'reasoning_tokens': 44}, 'prompt_time': 0.011467, 'prompt_tokens_details': None, 'queue_time': 0.149690775, 'total_time': 0.095953935}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerp

### Try it yourself

Ask:
- "I want the highest-rated science book under $20."
- "Do you have *Dune*?"
- "Give me two fantasy books under $25."

Watch what happens with *Dune*: since it isn't in `BOOKS`, a well-behaved
agent will call `search_books`/`book_details`, get "not found", and tell you
honestly rather than inventing a plausible-sounding answer. If it ever does
invent an answer, that's a sign the system prompt or model needs tightening.

### Key idea

The LLM decides **when** to search and **how** to communicate the answer.  
The Python tool owns the factual inventory.

# 2. Math Detective

Now give the agent several tools whose capabilities overlap slightly.

The interesting part is not the arithmetic. It is watching the model decide:

> "Which deterministic operation should I call to solve this request?"

### Why tools instead of letting the model "just calculate"?

Language models are notoriously unreliable at exact arithmetic, especially
with larger numbers, because they predict tokens rather than compute. By
giving the model tools like `calculate`, `is_prime`, `factor_integer`, and
`gcd`, every number that reaches the final answer has actually been computed
by real Python code — the model's job shifts from "compute the answer" to
"figure out which computation(s) answer the question, in what order."

This project also introduces **tool selection**: with four tools available,
the model has to read each one's docstring/description and figure out which
one (or combination) actually solves the user's question.

In [8]:
import ast
import math
import operator

# Map AST operator node types to the actual Python function that performs
# them. This whitelist is the core of the "safe eval" below: only these
# operators are permitted, so arbitrary code execution isn't possible.
_ALLOWED_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
    ast.USub: operator.neg,  # unary minus, e.g. -5
    ast.UAdd: operator.pos,  # unary plus, e.g. +5 (rare but valid)
}

def _safe_eval_node(node):
    """Recursively evaluate a parsed arithmetic expression tree.

    We never call Python's built-in eval()/exec() on model-provided text,
    because that would let a malicious or confused model run arbitrary code.
    Instead we parse the expression into an Abstract Syntax Tree (ast.parse)
    and manually walk it, only permitting numeric literals and the
    operators listed in _ALLOWED_OPS. Anything else (function calls, names,
    attribute access, etc.) raises an error instead of executing.
    """
    if isinstance(node, ast.Expression):
        return _safe_eval_node(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](
            _safe_eval_node(node.left),
            _safe_eval_node(node.right),
        )
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval_node(node.operand))
    raise ValueError("Unsupported expression.")

@tool
def calculate(expression: str) -> str:
    """Safely evaluate a basic arithmetic expression."""
    tree = ast.parse(expression, mode="eval")
    return str(_safe_eval_node(tree))

@tool
def is_prime(n: int) -> bool:
    """Return whether an integer is prime."""
    if n < 2:
        return False
    # Only need to test divisors up to sqrt(n): if n has a factor larger
    # than its square root, it must also have a corresponding factor
    # smaller than the square root, which we'd have already found.
    for d in range(2, math.isqrt(n) + 1):
        if n % d == 0:
            return False
    return True

@tool
def factor_integer(n: int) -> list[int]:
    """Return the prime factors of a positive integer."""
    if n < 1:
        raise ValueError("n must be positive")
    # Classic trial-division factorization: repeatedly divide out the
    # smallest possible factor until none remain, growing d until d*d > n.
    factors = []
    d = 2
    while d * d <= n:
        while n % d == 0:
            factors.append(d)
            n //= d
        d += 1
    if n > 1:
        # Whatever is left over after dividing out all smaller factors
        # must itself be a prime factor.
        factors.append(n)
    return factors

@tool
def gcd(a: int, b: int) -> int:
    """Return the greatest common divisor of two integers."""
    return math.gcd(a, b)

# One agent, four overlapping tools. The system prompt nudges the model to
# always reach for a tool rather than attempt non-trivial mental math.
math_agent = create_agent(
    model=MODEL,
    tools=[calculate, is_prime, factor_integer, gcd],
    system_prompt=(
        "You are a mathematical detective. "
        "Use tools for computations rather than doing non-trivial arithmetic mentally. "
        "Explain the result briefly."
    ),
)

In [9]:
ask(
    math_agent,
    "Is 221 prime? If not, factor it, and also tell me the gcd of 221 and 39."
)

**Is 221 prime?**  
No. 221 can be factored as  

\[
221 = 13 \times 17
\]

Both 13 and 17 are prime, so 221 is a composite number.

**GCD of 221 and 39**

Using the Euclidean algorithm:

1. \(221 \div 39 = 5\) remainder \(26\) → \(221 = 39 \times 5 + 26\)
2. \(39 \div 26 = 1\) remainder \(13\) → \(39 = 26 \times 1 + 13\)
3. \(26 \div 13 = 2\) remainder \(0\)

The last non‑zero remainder is \(13\). Therefore

\[
\gcd(221, 39) = 13
\]

So, 221 is not prime (its prime factors are 13 and 17), and the greatest common divisor of 221 and 39 is 13.


{'messages': [HumanMessage(content='Is 221 prime? If not, factor it, and also tell me the gcd of 221 and 39.', additional_kwargs={}, response_metadata={}, id='6908474a-2043-4cbf-a985-13a06e94fe28'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "We need to check if 221 is prime. 221 is 13*17? 13*17=221. So not prime. Factor: 13 and 17. gcd(221,39). Compute gcd. 221 mod 39? 39*5=195, remainder 26. Then gcd(39,26): 39 mod 26=13. 26 mod 13=0 => gcd=13. So gcd is 13. We'll use tool for factor_integer and gcd.", 'tool_calls': [{'id': 'fc_c5cfe1f8-85b6-4bfb-9532-063e6f608faf', 'function': {'arguments': '{"n":221}', 'name': 'factor_integer'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 129, 'prompt_tokens': 244, 'total_tokens': 373, 'completion_time': 0.141213748, 'completion_tokens_details': {'reasoning_tokens': 106}, 'prompt_time': 0.011966664, 'prompt_tokens_details': None, 'queue_time': 0.203312103, 'total_time': 0.153180412}, 'model_na

### Challenge

Try:

> A number has prime factorization `2^3 * 3 * 7`. What is the number, and is the number plus one prime?

This can make the agent chain multiple tool calls in one run: first compute
`2**3 * 3 * 7` with `calculate` (=168), then check `is_prime(169)`. Watching
this in the trace (e.g. via LangSmith) shows the model planning a short
sequence of deterministic steps rather than trying to do it all "in its head."

# 3. Bug Triage Desk — Structured Output

Agents are not always supposed to produce prose.

A backend often wants something machine-readable such as:

```json
{
  "severity": "...",
  "component": "...",
  "needs_human": true
}
```

LangChain agents can return a validated schema using `response_format=`.

### Why this is a different mode of agent

In Projects 1–2, the agent's final output was free text meant for a human to
read. Here, instead, we want the model's output to be a **typed, validated
object** we can plug straight into other code (a database row, an API
response, a routing decision).

`response_format=BugTriage` tells `create_agent` to constrain/validate the
model's final answer against the `BugTriage` Pydantic schema defined below,
instead of (or as well as) producing a plain chat message. If the model's
output doesn't fit the schema, LangChain will surface a validation problem
rather than silently accepting malformed data — this is what makes agent
output safe to hand to downstream systems without a human proofreading it
every time.

This project has **no tools** — it's a pure "read text in, produce
structured judgment out" agent, closer to a classification/extraction task
than to an autonomous agent.

In [10]:
from typing import Literal
from pydantic import BaseModel, Field

# Pydantic model = the schema the agent's final answer must conform to.
# `Literal[...]` restricts a field to an exact set of allowed string values,
# which LangChain turns into an enum-like constraint the model must respect
# (rather than being free to invent, e.g., "Severity: pretty bad").
class BugTriage(BaseModel):
    title: str = Field(description="Short normalized bug title")
    severity: Literal["low", "medium", "high", "critical"]
    component: Literal["frontend", "backend", "database", "auth", "infrastructure", "unknown"]
    summary: str
    reproduction_steps: list[str]
    needs_human: bool
    reason: str

# No `tools` here — this agent's only job is to read the bug report text
# and emit a `BugTriage` object. `response_format=BugTriage` is what wires
# in the schema validation.
triage_agent = create_agent(
    model=MODEL,
    tools=[],
    response_format=BugTriage,
    system_prompt=(
        "You triage software bug reports. "
        "Be conservative with 'critical': reserve it for major outage, data loss, "
        "security compromise, or similarly severe impact."
    ),
)

In [11]:
report = '''
After today's deployment, users can log in but clicking Export PDF spins
for around 30 seconds and then returns HTTP 500. CSV export still works.
I reproduced it on Chrome and Firefox.
'''

# Same invocation pattern as `ask()`, but called directly here because we
# want the raw `result` dict rather than just a printed final message.
result = triage_agent.invoke({
    "messages": [{"role": "user", "content": report}]
})

# When response_format is set, LangChain adds a "structured_response" key
# to the result containing a validated instance of our Pydantic model
# (in addition to the normal "messages" list).
triage = result["structured_response"]
print(triage)
print()
print(triage.model_dump())  # .model_dump() converts it to a plain dict/JSON-ready form

title='PDF export returns HTTP 500 after deployment' severity='high' component='backend' summary='After the latest deployment, clicking the Export PDF button results in a 30-second delay followed by an HTTP 500 error. CSV export remains functional.' reproduction_steps=['Log in to the application', 'Navigate to the export functionality', 'Click the Export PDF button', 'Wait for the response (~30 seconds)', 'Observe HTTP 500 error'] needs_human=True reason='The backend PDF generation endpoint is throwing an unhandled exception, likely due to changes in the deployment. Needs investigation to identify and fix the server-side error.'

{'title': 'PDF export returns HTTP 500 after deployment', 'severity': 'high', 'component': 'backend', 'summary': 'After the latest deployment, clicking the Export PDF button results in a 30-second delay followed by an HTTP 500 error. CSV export remains functional.', 'reproduction_steps': ['Log in to the application', 'Navigate to the export functionality', 'Cl

### Why this matters

This is much easier to feed into:
- a database,
- a ticketing workflow,
- a routing function,
- an evaluation pipeline,

than parsing arbitrary prose.

# 4. Tiny Escape Room

This project makes tool use visible by giving the model a tiny world to manipulate.

The important architectural split is:

- **World state:** ordinary Python
- **Actions:** tools
- **Planning / language:** the model

This is a miniature version of many enterprise agents.

### Why this matters more than it looks

In Project 1, the tools were read-only (searching a catalog). Here, the tools
**mutate state** (`ROOM`), and — crucially — each tool enforces its own
preconditions in plain Python:

- `open_drawer` only reveals the key if the drawer isn't already open.
- `take_item` only succeeds for "brass key" *and* only if the drawer is open.
- `unlock_exit` only succeeds if the key is in the player's inventory.
- `leave_room` only succeeds if the door is unlocked.

The model can *say* anything it wants, but it can only *change the world* by
calling these functions, and each function independently re-checks whether
the action is actually valid. This means the model cannot skip a step or
"talk its way" into having escaped — every claim of success is backed by a
real state change checked by deterministic code. This is the same principle
production agents use to avoid a model incorrectly claiming that a payment
was processed, an email was sent, or a record was deleted when it wasn't.

In [12]:
# Global mutable "world state" for the escape room. Ordinary Python owns
# this dict; the model never edits it directly, only via the tools below.
ROOM = {
    "location": "study",
    "inventory": [],
    "drawer_open": False,
    "door_unlocked": False,
    "escaped": False,
}

def reset_room():
    """Reset the world to its initial state (handy for re-running demos)."""
    ROOM.update({
        "location": "study",
        "inventory": [],
        "drawer_open": False,
        "door_unlocked": False,
        "escaped": False,
    })

@tool
def inspect_room() -> str:
    """Inspect the current room and visible objects."""
    # This tool never changes state — it just reports a description,
    # optionally including facts that depend on current state (drawer open).
    observations = [
        "You are in a study.",
        "There is a locked exit door.",
        "There is a wooden desk with a drawer.",
        "A note says: 'The key hides where ideas are written.'",
    ]
    if ROOM["drawer_open"]:
        observations.append("The desk drawer is open.")
    return " ".join(observations)

@tool
def open_drawer() -> str:
    """Open the desk drawer."""
    if ROOM["drawer_open"]:
        return "The drawer is already open."
    ROOM["drawer_open"] = True  # <-- the actual, only, source of truth changes here
    return "The drawer opens. Inside is a small brass key."

@tool
def take_item(item: str) -> str:
    """Take an item that is currently accessible."""
    # Precondition check: the key can only be taken if the drawer has
    # already been opened. This prevents the model from "picking up"
    # something it hasn't actually revealed yet.
    if item.lower() == "brass key" and ROOM["drawer_open"]:
        if "brass key" not in ROOM["inventory"]:
            ROOM["inventory"].append("brass key")
        return "You now have the brass key."
    return f"You cannot take {item}."

@tool
def unlock_exit() -> str:
    """Unlock the exit door if the player has the right item."""
    # Another precondition check: no key in inventory -> door stays locked,
    # no matter what the model claims in its own reasoning/text.
    if "brass key" not in ROOM["inventory"]:
        return "The door remains locked. You need a key."
    ROOM["door_unlocked"] = True
    return "The brass key unlocks the exit door."

@tool
def leave_room() -> str:
    """Leave through the exit if it is unlocked."""
    if not ROOM["door_unlocked"]:
        return "You cannot leave; the exit is locked."
    ROOM["escaped"] = True
    return "You step outside. You escaped!"

# The agent is given the full set of action tools and told to rely on their
# results rather than assuming an action worked.
escape_agent = create_agent(
    model=MODEL,
    tools=[inspect_room, open_drawer, take_item, unlock_exit, leave_room],
    system_prompt=(
        "You are controlling a character in a tiny escape room. "
        "Your goal is to escape. Use tools to inspect and act. "
        "Do not claim an action succeeded unless a tool confirms it."
    ),
)

In [13]:
reset_room()  # make sure we start from a clean, locked, empty-inventory state

result = ask(
    escape_agent,
    "Escape from the room. Think through what you need to inspect and do."
)

# Print the ground-truth world state after the run so we can verify the
# agent actually changed it (rather than just claiming success in text).
print("\nFinal deterministic world state:")
pprint(ROOM)

You’ve successfully escaped the study. Good job!

Final deterministic world state:
{'door_unlocked': True,
 'drawer_open': True,
 'escaped': True,
 'inventory': ['brass key'],
 'location': 'study'}


### Discussion

This simple example demonstrates a useful rule:

> Let the LLM choose actions, but let deterministic software decide what those actions actually do.

The model cannot “hallucinate” the door into being unlocked because `leave_room()` checks Python state.

# 5. Study Coach with Short-Term Memory

Without a checkpointer, every invocation is effectively a fresh conversation unless you manually provide history.

With `InMemorySaver`, LangChain can preserve the message history for a `thread_id`.

This is **short-term conversational memory**, not a permanent user-profile database.

### What's actually happening under the hood

`create_agent` is built on LangGraph, which models the agent as a stateful
graph. A **checkpointer** is a pluggable component that saves/restores that
graph's state (here: the running message list) keyed by a `thread_id`.

- `InMemorySaver()` keeps this state in a plain Python dict in the current
  process — it disappears when the kernel restarts. In production you'd
  swap in a persistent checkpointer (e.g. backed by Postgres or Redis)
  without changing any other agent code.
- Every call to `agent.invoke(..., config=study_thread)` with the same
  `thread_id` automatically loads the prior messages for that thread,
  appends the new user message, runs the agent, and saves the updated
  history back — you never manually pass conversation history yourself.
- Using a *different* `thread_id` is how you isolate separate
  conversations/users from each other on the same agent instance.

In [14]:
from langgraph.checkpoint.memory import InMemorySaver

study_agent = create_agent(
    model=MODEL,
    tools=[],
    system_prompt=(
        "You are a concise study coach. "
        "Remember facts the learner tells you earlier in the same thread. "
        "Ask at most one question at a time."
    ),
    # Attaching a checkpointer is the one line that turns a stateless agent
    # into one with conversational memory, scoped per thread_id.
    checkpointer=InMemorySaver(),
)

# `config` dicts in LangGraph carry run-time metadata; "thread_id" is the
# specific key the checkpointer uses to group messages into one conversation.
study_thread = {"configurable": {"thread_id": "student-demo-1"}}

In [15]:
# First turn on this thread: introduce two facts (what confuses the learner,
# what they already understand). These get saved into thread "student-demo-1".
ask(
    study_agent,
    "I'm studying probability. I understand random variables, but conditional expectation confuses me.",
    config=study_thread,
)

Conditional expectation is just the *expected value of a random variable when you “freeze” the information in another variable*.  
Formally, for random variables \(X\) and \(Y\)

\[
\mathbb{E}[X\mid Y] \quad\text{is the random variable } g(Y)
\]

such that for every event \(A\) measurable with respect to \(Y\),

\[
\mathbb{E}[X\mathbf{1}_A] = \mathbb{E}[\,g(Y)\mathbf{1}_A\,].
\]

In practice, you treat \(Y\) as a fixed value, compute the usual expectation of \(X\) under that condition, and then let \(Y\) vary.  
**Example (discrete):**  
Let \(X\) be the number of heads in two coin flips and let \(Y\) be the result of the first flip (\(Y=0\) for tail, \(Y=1\) for head).  
* If \(Y=0\), the second flip is the only random part, so \(\mathbb{E}[X\mid Y=0]=0.5\).  
* If \(Y=1\), then \(X\) is at least 1, and the expected total is \(1+0.5=1.5\).  
Thus \(\mathbb{E}[X\mid Y]=0.5\mathbf{1}_{\{Y=0\}}+1.5\mathbf{1}_{\{Y=1\}}\).

Does this example help, or would you like to see a different type 

{'messages': [HumanMessage(content="I'm studying probability. I understand random variables, but conditional expectation confuses me.", additional_kwargs={}, response_metadata={}, id='a1586aa4-0cfc-4e1c-92e8-e35aa44d6a9b'),
  AIMessage(content='Conditional expectation is just the *expected value of a random variable when you “freeze” the information in another variable*.  \nFormally, for random variables \\(X\\) and \\(Y\\)\n\n\\[\n\\mathbb{E}[X\\mid Y] \\quad\\text{is the random variable } g(Y)\n\\]\n\nsuch that for every event \\(A\\) measurable with respect to \\(Y\\),\n\n\\[\n\\mathbb{E}[X\\mathbf{1}_A] = \\mathbb{E}[\\,g(Y)\\mathbf{1}_A\\,].\n\\]\n\nIn practice, you treat \\(Y\\) as a fixed value, compute the usual expectation of \\(X\\) under that condition, and then let \\(Y\\) vary.  \n**Example (discrete):**  \nLet \\(X\\) be the number of heads in two coin flips and let \\(Y\\) be the result of the first flip (\\(Y=0\\) for tail, \\(Y=1\\) for head).  \n* If \\(Y=0\\), the se

In [16]:
# Second turn, same thread_id: no facts are repeated here, yet the agent
# should still recall them, because the checkpointer restored the earlier
# messages for "student-demo-1" before this call ran.
ask(
    study_agent,
    "What topic did I say confuses me, and what prerequisite did I say I already understand?",
    config=study_thread,
)

You mentioned that conditional expectation is the part that confuses you, and that you already understand random variables.


{'messages': [HumanMessage(content="I'm studying probability. I understand random variables, but conditional expectation confuses me.", additional_kwargs={}, response_metadata={}, id='a1586aa4-0cfc-4e1c-92e8-e35aa44d6a9b'),
  AIMessage(content='Conditional expectation is just the *expected value of a random variable when you “freeze” the information in another variable*.  \nFormally, for random variables \\(X\\) and \\(Y\\)\n\n\\[\n\\mathbb{E}[X\\mid Y] \\quad\\text{is the random variable } g(Y)\n\\]\n\nsuch that for every event \\(A\\) measurable with respect to \\(Y\\),\n\n\\[\n\\mathbb{E}[X\\mathbf{1}_A] = \\mathbb{E}[\\,g(Y)\\mathbf{1}_A\\,].\n\\]\n\nIn practice, you treat \\(Y\\) as a fixed value, compute the usual expectation of \\(X\\) under that condition, and then let \\(Y\\) vary.  \n**Example (discrete):**  \nLet \\(X\\) be the number of heads in two coin flips and let \\(Y\\) be the result of the first flip (\\(Y=0\\) for tail, \\(Y=1\\) for head).  \n* If \\(Y=0\\), the se

### Experiment

Change the thread:

```python
other_thread = {"configurable": {"thread_id": "student-demo-2"}}
```

Ask the same follow-up question with `other_thread`.

The second conversation should not inherit the first thread's history,
because `InMemorySaver` stores history per `thread_id` — a fresh id means a
fresh, empty history, exactly like a new user starting a new chat session.

# 6. Guarded Support Agent — Middleware

Middleware wraps the agent loop.

Here we implement a deliberately simple `before_model` hook that limits how much conversation history is sent to the model.

This is useful because real agents often need policy around the model, not just a prompt inside the model.

### Why trim history at all?

As a conversation grows, the message list sent to the model on every turn
grows too — this costs more tokens (money and latency) and can eventually
exceed the model's context window. A `before_model` middleware hook runs
right before each model call and can rewrite the state (e.g. drop old
messages) without the model or the rest of the agent code needing to know
about it.

This is a **policy enforced by software**, not by asking the model nicely to
"please keep things brief" — the same category of guarantee that made the
Escape Room's tools reliable in Project 4, just applied to the model call
itself rather than to world state.

In [17]:
from typing import Any
from langchain.agents import AgentState
from langchain.agents.middleware import before_model
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime

# The @before_model decorator registers this function to run immediately
# before every call to the underlying chat model, with the chance to
# rewrite the agent's state (here, `state["messages"]`) beforehand.
@before_model
def keep_recent_context(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    Keep the first message plus a small recent window.

    This is intentionally tiny for demonstration purposes.
    Production trimming should usually be token-aware.
    """
    messages = state["messages"]

    # If the conversation is still short, do nothing: returning None means
    # "no change" and the model sees the full history as normal.
    if len(messages) <= 6:
        return None

    # Keep the very first message (often carries important initial context,
    # like the user's opening description of their project) plus only the
    # last 5 messages, dropping everything in between.
    first_message = messages[0]
    recent_messages = messages[-5:]

    # LangGraph message-state updates are additive by default, so to
    # actually *replace* the history we must explicitly clear it first
    # (RemoveMessage with the REMOVE_ALL_MESSAGES sentinel) and then supply
    # the trimmed list we want to keep.
    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            first_message,
            *recent_messages,
        ]
    }

support_agent = create_agent(
    model=MODEL,
    tools=[],
    system_prompt=(
        "You are a concise support assistant. "
        "Answer based on the conversation context you are given."
    ),
    # Middleware list is applied around the agent loop; multiple hooks can
    # be composed here (before_model, after_model, tool-call guards, etc.).
    middleware=[keep_recent_context],
    checkpointer=InMemorySaver(),
)

support_thread = {"configurable": {"thread_id": "support-demo-1"}}

In [18]:
# Feed in seven separate user turns on the same thread. By the time we get
# past 6 total messages accumulated, keep_recent_context starts trimming
# the history the model actually sees (though the checkpointer still stores
# the full, untrimmed conversation).
messages = [
    "My project is called Orion.",
    "It is written in Python.",
    "The API is built with FastAPI.",
    "The database is PostgreSQL.",
    "We deploy it with Docker.",
    "The symptom is that requests sometimes time out.",
    "The timeout appears only after deployment.",
]

for message in messages:
    ask(support_agent, message, config=support_thread)

# By now, older facts (e.g. "written in Python") may have been trimmed out
# of what the model sees, while the very first message and the most recent
# ones are still visible — demonstrating the middleware's trimming policy
# in action.
ask(
    support_agent,
    "Summarize the technical context you can still see.",
    config=support_thread,
)

Got it—your project is named Orion. How can I help you with it?
Great! Orion is a Python project. What do you need help with? (e.g., setup, debugging, features, deployment, etc.)
Nice—Orion is a FastAPI app in Python.  
What do you need help with? (e.g., project setup, routing, deployment, testing, etc.)
Got it—Orion uses FastAPI with a PostgreSQL backend.  
How can I help you with it?
Sounds great—FastAPI + PostgreSQL in Docker.  
What specifically would you like to set up, troubleshoot, or improve?
**Possible causes & quick checks**

| Area | What to look for | Quick fix / test |
|------|------------------|------------------|
| **Container resources** | CPU/memory limits in Docker Compose/Swarm/K8s | Increase limits; run `docker stats` to see if containers hit 100 % CPU or swap |
| **Database connection pool** | Too many open connections or long‑running queries | Check `pg_stat_activity`; adjust `max_connections`, use async `asyncpg` pool, add `timeout` to queries |
| **Network laten

{'messages': [HumanMessage(content='My project is called Orion.', additional_kwargs={}, response_metadata={}, id='39741366-b517-47e0-ab8c-bf8a9bfd368a'),
  HumanMessage(content='The symptom is that requests sometimes time out.', additional_kwargs={}, response_metadata={}, id='b84dbe2a-4009-4f68-94ec-3c724cc0f8a3'),
  AIMessage(content='**Possible causes & quick checks**\n\n| Area | What to look for | Quick fix / test |\n|------|------------------|------------------|\n| **Container resources** | CPU/memory limits in Docker Compose/Swarm/K8s | Increase limits; run `docker stats` to see if containers hit 100\u202f% CPU or swap |\n| **Database connection pool** | Too many open connections or long‑running queries | Check `pg_stat_activity`; adjust `max_connections`, use async `asyncpg` pool, add `timeout` to queries |\n| **Network latency** | Docker bridge or overlay network delays | Run `docker network inspect`; try `ping`/`traceroute` between containers |\n| **Timeout settings** | FastAPI

### What middleware changes conceptually

A system prompt says:

> “Please behave this way.”

Middleware says:

> “The software around the model will enforce or transform something.”

Middleware is therefore a natural place for concerns such as:
- context management,
- retries,
- logging,
- PII handling,
- tool policies,
- human approval,
- model routing.

The current LangChain middleware API exposes hooks around the agent loop rather than requiring you to rewrite the loop yourself.

# 7. Put the pieces together

| Capability | LangChain idea | Project |
|---|---|---|
| Model + harness | `create_agent` | Bookstore |
| Agent chooses deterministic actions | tools | Bookstore / Math / Escape Room |
| Machine-readable result | `response_format` | Bug Triage |
| Stateful environment | Python state + tools | Escape Room |
| Conversation persistence | checkpointer + `thread_id` | Study Coach |
| Control around the loop | middleware | Guarded Support |

## A useful mental model

```text
                         ┌────────────────────┐
User ──────────────────►│     Agent State    │
                         └─────────┬──────────┘
                                   │
                                   ▼
                         ┌────────────────────┐
                         │     Middleware     │
                         └─────────┬──────────┘
                                   │
                                   ▼
                         ┌────────────────────┐
                         │       Model        │
                         └─────────┬──────────┘
                                   │
                         tool call?│
                      ┌────────────┴────────────┐
                      ▼                         ▼
             ┌────────────────┐        ┌────────────────┐
             │ Python Tool(s) │        │  Final Answer  │
             └───────┬────────┘        └────────────────┘
                     │
                     └──── result back to model
```

Reading the diagram left to right / top to bottom: a user message enters the
agent's state, optional middleware may transform that state (e.g. trimming
history, as in Project 6), the model then reasons over what it's given. If it
decides it needs data or wants to take an action, it emits a tool call, which
the harness routes to the matching Python function; the tool's return value
is fed back into the state and the model is invoked again. This loop repeats
until the model produces a final answer instead of a tool call — that's what
`agent.invoke()` returns.

The key idea is that an agent is not "an LLM that can do everything."

It is an LLM embedded inside a software system whose data, actions, state, validation, safety, and control flow can still be ordinary deterministic software.

# 8. Extensions for students

These are good next exercises without adding much infrastructure:

1. **Bookstore:** add `reserve_book(title, customer_name)` and make inventory mutable.
2. **Math Detective:** add a statistics tool that computes mean, variance, and quantiles.
3. **Bug Triage:** route high/critical issues into separate Python handler functions.
4. **Escape Room:** add a second room and a misleading object.
5. **Study Coach:** add a tool that serves questions from a hard-coded quiz bank.
6. **Middleware:** log the number of model/tool calls or add a human-approval step for one sensitive tool.

A strong class exercise is:

> “Which decisions should belong to the model, and which decisions should remain deterministic code?”

That question is more important than memorizing individual LangChain APIs.